In [1]:
import os

import pandas as pd

In [2]:
MERGED_PATH = "../data/processed/metadata/mergedCatalogue.txt"
BOUNDS_PATH = "../data/processed/metadata/curriculum/sizeClassBounds.csv"
BAD_BUBBLES_PATH = "../data/processed/metadata/badBubbles.txt"

OUTPUT_PATH = "../data/processed/metadata/cleanedSegregatedMergedCatalogue.txt"

SIZE_CLASSES = ["tiny", "small", "medium", "large", "giants", "edgeLarge"]

CURRICULUM_STAGES = {
    "stage00": ["edgeLarge"],
    "stage01": ["giants", "edgeLarge"],
    "stage02": ["large", "giants", "edgeLarge"],
    "stage03": ["medium", "large", "giants", "edgeLarge"],
    "stage04": ["small", "medium", "large", "giants", "edgeLarge"],
    "stage05": ["tiny", "small", "medium", "large", "giants", "edgeLarge"],
}

In [3]:
df = pd.read_csv(MERGED_PATH)
df.columns = df.columns.str.strip()
df["GLOBAL_ID"] = df["GLOBAL_ID"].astype(int)

assert df["GLOBAL_ID"].is_unique, "GLOBAL_ID is not unique in the merged catalogue"

bounds = pd.read_csv(BOUNDS_PATH).sort_values("lower_bound_pc").reset_index(drop=True)

assert list(bounds["SIZE_CLASS"]) == SIZE_CLASSES, (
    f"sizeClassBounds.csv order {list(bounds['SIZE_CLASS'])} does not match "
    f"{SIZE_CLASSES} - re-run bubbleStrattification.ipynb"
)

print("Merged catalogue bubbles:", len(df))
print()
print(bounds.to_string(index=False))

Merged catalogue bubbles: 3318

SIZE_CLASS  lower_bound_pc  upper_bound_pc
      tiny            -inf           18.00
     small           18.00           24.00
    medium           24.00           30.00
     large           30.00           41.00
    giants           41.00          110.21
 edgeLarge          110.21             inf


In [4]:
bin_edges = list(bounds["lower_bound_pc"]) + [bounds["upper_bound_pc"].iloc[-1]]

df["SIZE_CLASS"] = pd.cut(
    df["AVG_RAD_PC"],
    bins=bin_edges,
    labels=SIZE_CLASSES,
    right=False,
).astype(str)

unassigned = (df["SIZE_CLASS"] == "nan").sum()
if unassigned:
    raise ValueError(f"{unassigned} bubbles could not be assigned to a size class.")

print("Size classes assigned to all", len(df), "bubbles:")
print(df["SIZE_CLASS"].value_counts().reindex(SIZE_CLASSES).to_string())

Size classes assigned to all 3318 bubbles:
SIZE_CLASS
tiny         378
small        592
medium       667
large        790
giants       778
edgeLarge    113


In [5]:
if os.path.exists(BAD_BUBBLES_PATH):
    bad_df = pd.read_csv(BAD_BUBBLES_PATH)
    bad_df.columns = bad_df.columns.str.strip()

    if "GLOBAL_ID" not in bad_df.columns:
        raise ValueError(
            f"badBubbles.txt has columns {list(bad_df.columns)}; expected a single "
            "GLOBAL_ID column. The archived CATALOGUE,ID format is not supported."
        )

    bad_df["GLOBAL_ID"] = bad_df["GLOBAL_ID"].astype(int)
    bad_ids = bad_df.drop_duplicates(subset=["GLOBAL_ID"])["GLOBAL_ID"]

    matched = bad_ids.isin(df["GLOBAL_ID"])

    print("Bad bubbles listed :", len(bad_df))
    print("Unique             :", len(bad_ids))
    print("Matching catalogue :", int(matched.sum()))
    print("Unmatched          :", int((~matched).sum()))
    if (~matched).any():
        print("  unmatched ids:", sorted(bad_ids[~matched].tolist()))
else:
    bad_ids = pd.Series([], dtype=int, name="GLOBAL_ID")
    print(f"No {BAD_BUBBLES_PATH} - skipping the cleaning step.")
    print("Run the manual cleaning against outputs/figures/catalogueMaps/ first "
          "if you expected bubbles to be removed.")

Bad bubbles listed : 766
Unique             : 617
Matching catalogue : 616
Unmatched          : 1
  unmatched ids: [3734]


In [6]:
df_clean = df[~df["GLOBAL_ID"].isin(bad_ids)].copy()

removed = len(df) - len(df_clean)

print("Total bubbles before cleaning:", len(df))
print("Total bubbles after cleaning :", len(df_clean))
print("Removed bubbles              :", removed)

assert df_clean["GLOBAL_ID"].is_unique, "duplicate GLOBAL_ID survived cleaning"
assert removed == int(bad_ids.isin(df["GLOBAL_ID"]).sum()), (
    "removed count does not match the matched bad-bubble count"
)

Total bubbles before cleaning: 3318
Total bubbles after cleaning : 2702
Removed bubbles              : 616


In [7]:

REFERENCE_COUNTS = {
    "tiny": 291, "small": 436, "medium": 528,
    "large": 674, "giants": 686, "edgeLarge": 87,
}

counts = df_clean["SIZE_CLASS"].value_counts().reindex(SIZE_CLASSES)

check = pd.DataFrame({
    "count": counts,
    "reference": pd.Series(REFERENCE_COUNTS).reindex(SIZE_CLASSES),
})
check["delta"] = check["count"] - check["reference"]

print(check.to_string())
print(f"\ntotal {int(check['count'].sum())} vs reference {int(check['reference'].sum())}")

if (check["delta"] != 0).any():
    print("\nNOTE: counts differ from the old chain. Expected only if badBubbles.txt "
          "or the bounds changed - otherwise investigate before continuing.")
else:
    print("\nMatches the old three-notebook chain exactly.")

           count  reference  delta
tiny         291        291      0
small        436        436      0
medium       528        528      0
large        674        674      0
giants       686        686      0
edgeLarge     87         87      0

total 2702 vs reference 2702

Matches the old three-notebook chain exactly.


In [8]:
print(f"{'stage':<10}{'bubbles':>9}   classes")
for stage, classes in CURRICULUM_STAGES.items():
    n = int(df_clean["SIZE_CLASS"].isin(classes).sum())
    print(f"{stage:<10}{n:>9}   {', '.join(classes)}")

stage       bubbles   classes
stage00          87   edgeLarge
stage01         773   giants, edgeLarge
stage02        1447   large, giants, edgeLarge
stage03        1975   medium, large, giants, edgeLarge
stage04        2411   small, medium, large, giants, edgeLarge
stage05        2702   tiny, small, medium, large, giants, edgeLarge


In [10]:
df_reduced = df_clean.drop(['ARM', 'DIST_ARM_PC', 'GAL_RAD_KPC'], axis=1)

In [12]:
df_reduced.to_csv(OUTPUT_PATH, index=False)

print("Saved cleaned, size-labelled catalogue to", OUTPUT_PATH)
print("Columns:", list(df_reduced.columns))

Saved cleaned, size-labelled catalogue to ../data/processed/metadata/cleanedSegregatedMergedCatalogue.txt
Columns: ['GLOBAL_ID', 'ID', 'RA_DMS', 'DEC_DMS', 'SEMI_MAJ_PC', 'SEMI_MIN_PC', 'AVG_RAD_PC', 'PA_DEG', 'CATALOGUE', 'SIZE_CLASS']
